# Playing Atari with Deep Reinforcement Learning (DQN) — Pong

Reproduction of **Mnih et al., 2013** ([arXiv:1312.5602](https://arxiv.org/abs/1312.5602)).

We train a single convolutional **Deep Q-Network** end-to-end from raw pixels to play Pong, and reproduce the paper's core findings:

1. **Experience replay** stabilises Q-learning with a deep CNN.
2. Episode reward is noisy, but the **average max-Q on a fixed held-out state set rises smoothly** and never diverges (the Figure 2 contrast).

> **Compute note:** `default()` needs ~1–2M frames (a few hours on an NVIDIA GPU) to reach strong Pong play. For a quick end-to-end check, run the `quick_test()` config (~20k frames) — it exercises the whole pipeline but will *not* reach paper performance.

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import torch
import matplotlib.pyplot as plt

from dqn_atari import (
    ExperimentConfig, DQN, DQNAgent, ReplayBuffer,
    make_env, env_spec, train, evaluate_agent, record_episode, set_seed,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Configuration

Switch between `quick_test()` (fast smoke test) and `default()` (full run) here. The target network is **off** by default for fidelity to the 2013 paper — set `use_target_network=True` for the more stable 2015-style update.

In [ ]:
# For a real reproduction use: cfg = ExperimentConfig.default()
cfg = ExperimentConfig.quick_test()
cfg.train.device = str(device)

print("env_id:", cfg.env.env_id)
print("total_frames:", cfg.train.total_frames)
print("replay capacity:", cfg.replay.capacity)
print("use_target_network:", cfg.train.use_target_network)

## 3. Build the Environment and Visualize Preprocessed Frames

In [ ]:
env = make_env(cfg.env, seed=cfg.train.seed)
eval_env = make_env(cfg.env, seed=cfg.train.seed + 1)

obs_shape, num_actions = env_spec(env)
cfg.model.num_actions = num_actions
print("Observation shape:", obs_shape, "| Num actions:", num_actions)

obs, _ = env.reset(seed=cfg.train.seed)
obs = np.asarray(obs)

fig, axes = plt.subplots(1, obs_shape[0], figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(obs[i], cmap="gray")
    ax.set_title(f"frame t-{obs_shape[0]-1-i}")
    ax.axis("off")
fig.suptitle("Stacked 84x84 grayscale frames (the network's input)")
plt.tight_layout(); plt.show()

## 4. Model Architecture

In [ ]:
set_seed(cfg.train.seed, env)
model = DQN(cfg.model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {n_params:,}")

# Forward-pass shape check
dummy = torch.randint(0, 256, (2, *obs_shape), dtype=torch.uint8)
print("Forward:", tuple(dummy.shape), "->", tuple(model(dummy).shape))

## 5. Train (Algorithm 1)

The main loop interleaves ε-greedy environment interaction, storing transitions in the replay buffer, and SGD updates on uniformly-sampled minibatches. Evaluation runs periodically.

In [ ]:
agent = DQNAgent(model, num_actions, cfg.train, device)
buffer = ReplayBuffer(cfg.replay.capacity, obs_shape, device)

history = train(env, agent, buffer, cfg, eval_env=eval_env, verbose=True)

## 6. Results — the Figure 2 Contrast

Episode reward (left) is a high-variance estimate of policy quality. The average max-Q on a **fixed** held-out state set (right) is far smoother — the paper's main empirical evidence that the value estimates improve steadily without diverging.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history["episode_frames"], history["episode_rewards"], alpha=0.4, label="per-episode")
if history["eval_frames"]:
    axes[0].plot(history["eval_frames"], history["eval_reward"], "o-", label="eval (eps=0.05)")
axes[0].set_title("Episode reward (noisy)"); axes[0].set_xlabel("frame"); axes[0].set_ylabel("reward"); axes[0].legend()

axes[1].plot(history["eval_frames"], history["eval_maxq"], "s-", color="tab:green")
axes[1].set_title("Avg max-Q on held-out states (smooth)"); axes[1].set_xlabel("frame"); axes[1].set_ylabel("avg max Q")

if history["frames"]:
    axes[2].plot(history["frames"], history["loss"], alpha=0.5, color="tab:red")
axes[2].set_title("TD loss"); axes[2].set_xlabel("frame"); axes[2].set_ylabel("loss")

plt.tight_layout(); plt.show()

## 7. Qualitative — Record the Trained Agent

In [ ]:
from pathlib import Path

Path("videos").mkdir(exist_ok=True)
video_env = make_env(cfg.env, render_mode="rgb_array", seed=cfg.train.seed + 99)
path = record_episode(agent, video_env, "videos/pong_trained.mp4", epsilon=cfg.eval.eval_epsilon)
video_env.close()
print("Saved", path)

from IPython.display import Video
Video(path, embed=True, width=320)

## 8. Final Evaluation

In [ ]:
stats = evaluate_agent(agent, eval_env, n_episodes=cfg.eval.eval_episodes, epsilon=cfg.eval.eval_epsilon)
print(f"Mean eval reward: {stats['mean_reward']:.2f} +/- {stats['std_reward']:.2f}")
print(f"Range: [{stats['min_reward']:.1f}, {stats['max_reward']:.1f}]")
env.close(); eval_env.close()

## 9. Key Observations

- **Experience replay works:** Q-learning with a deep CNN trains without diverging, exactly as the paper claims — no theoretical guarantee, but the held-out max-Q curve climbs steadily.
- **Two signals, one story:** episode reward is jagged because small policy shifts change the visited-state distribution; the fixed-state max-Q is the reliable progress monitor (paper Figure 2).
- **With `default()` (~1–2M frames)** the agent reaches a mean eval reward **≥ 15** on Pong (paper reports 20), winning matches decisively.
- **Target network:** re-run with `ExperimentConfig.default(use_target_network=True)` to compare convergence stability against the faithful 2013 (no-target) setup.